# 🛒 Customer Churn Prediction Pipeline

**A complete, reusable binary classification pipeline — from raw transactions to business-ready predictions.**

---

## What this notebook does
| Step | What happens |
|------|-------------|
| 1 | Configure all settings in one place |
| 2 | Load & validate your transaction data |
| 3 | Engineer 10 customer-level features |
| 4 | Split into train / test sets |
| 5 | Train 3 models with cross-validation |
| 6 | Evaluate all models on the test set |
| 7 | Score every customer + assign risk tiers |
| 8 | Save models and predictions to disk |
| 9 | Generate the 9-panel visual dashboard |
| 10 | Print the final business summary |

---

## ♻️  Reusability — adapting to your own data

This notebook is designed to work on **any transaction dataset**, not just Online Retail.
To adapt it:

1. **Change `SETTINGS` (Cell 2)** — point `DATA_FILE` to your CSV, set `DATE_COLUMN`,
   `CUSTOMER_ID_COL`, `INVOICE_COL`, and `REVENUE_COL` to match your column names.
2. **Review `FEATURE_COLUMNS`** — keep, remove, or add features that suit your domain.
3. **Adjust `CHURN_THRESHOLD_DAYS`** — set the inactivity window that makes business sense.
4. **Run all cells top to bottom.**

> ⚠️ **Data leakage warning:** Never add `Recency` to `FEATURE_COLUMNS`.
> The churn label IS defined as `Recency > threshold` — including it gives the model
> the answer, not a pattern to learn.

---
## Cell 1 — Imports
All libraries loaded once here. If any `pip install` is missing, run:
```
pip install pandas scikit-learn xgboost matplotlib seaborn joblib
```

In [ ]:
# =============================================================================
# IMPORTS
# Standard Python libraries + the three ML/data-science stacks.
# =============================================================================

import json
import time
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
print("✅  All libraries imported successfully.")

---
## Cell 2 — Settings
**This is the only cell you need to change to adapt the notebook to a new dataset.**

| Variable | What it controls |
|----------|-----------------|
| `DATA_FILE` | Path to your CSV file |
| `DATE_COLUMN` | The column containing purchase/transaction dates |
| `CUSTOMER_ID_COL` | The column that uniquely identifies each customer |
| `INVOICE_COL` | The column that identifies each transaction/order |
| `REVENUE_COL` | The column containing revenue/spend per line |
| `CHURN_THRESHOLD_DAYS` | Days of inactivity that defines a churned customer |
| `FEATURE_COLUMNS` | The 10 behavioural features the model will learn from |

In [ ]:
# =============================================================================
# SETTINGS — change these to adapt the notebook to any transaction dataset
# =============================================================================

# ── Your data ─────────────────────────────────────────────────────────────────
DATA_FILE        = "data/raw/cleaned_online_retail.csv"  # ← path to your CSV
DATE_COLUMN      = "InvoiceDate"     # ← date/time column in your CSV
CUSTOMER_ID_COL  = "CustomerID"      # ← unique customer identifier column
INVOICE_COL      = "InvoiceNo"       # ← unique transaction/order identifier
REVENUE_COL      = "Revenue"         # ← revenue per line-item column
QUANTITY_COL     = "Quantity"        # ← quantity per line-item column
PRODUCT_COL      = "StockCode"       # ← product/SKU identifier column
COUNTRY_COL      = "Country"         # ← customer country column

# ── Churn definition ──────────────────────────────────────────────────────────
# BUSINESS DECISION: how many days without a purchase = churned?
# Lowering this makes the definition stricter (more customers flagged).
# Raising it makes it more lenient (fewer customers flagged).
CHURN_THRESHOLD_DAYS = 90

# ── Features the model learns from ───────────────────────────────────────────
# Each feature summarises a different aspect of customer behaviour.
# Recency is INTENTIONALLY excluded — it would cause direct data leakage.
FEATURE_COLUMNS = [
    "TotalRevenue",        # total money spent (customer lifetime value proxy)
    "TotalOrders",         # number of distinct shopping trips
    "TotalItems",          # total units purchased
    "AvgLineItemRevenue",  # average revenue per product line (not per order)
    "BasketValue",         # average spend per trip  = TotalRevenue / TotalOrders
    "RevenuePerItem",      # average unit price paid = TotalRevenue / TotalItems
    "UniqueProducts",      # breadth of product engagement
    "UniqueCountries",     # number of shipping destinations
    "Tenure",              # days from first → last purchase (loyalty lifespan)
    "AvgDaysBetween",      # typical inter-purchase gap (purchase frequency proxy)
]

# ── Risk tier thresholds ──────────────────────────────────────────────────────
# The best model's churn probability (0.0→1.0) is bucketed into these tiers.
# Adjust bins to match your retention team's capacity.
RISK_BINS   = [0.0, 0.33, 0.66, 1.0]
RISK_LABELS = ["Low Risk", "Medium Risk", "High Risk"]

# ── Train / test split ────────────────────────────────────────────────────────
TEST_SIZE    = 0.20  # 20% held out for final unbiased evaluation
RANDOM_STATE = 42    # makes every run produce identical results
CV_FOLDS     = 5     # number of cross-validation folds

# ── Model hyperparameters ─────────────────────────────────────────────────────
# class_weight="balanced" tells sklearn to upweight the minority (churned) class
# so it does not get drowned out by the majority (active) class.
LR_PARAMS = dict(
    max_iter=1000,          # max solver iterations before giving up
    class_weight="balanced",
    random_state=RANDOM_STATE,
    C=1.0,                  # regularisation strength (higher = less penalty)
)
RF_PARAMS = dict(
    n_estimators=300,       # number of decision trees in the forest
    max_depth=10,           # max depth per tree (limits overfitting)
    min_samples_leaf=5,     # min samples required at each leaf node
    max_features="sqrt",    # features considered per split (√n_features)
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,              # use all CPU cores
)
XGB_PARAMS = dict(
    n_estimators=300,       # boosting rounds
    max_depth=4,            # shallow trees work best for boosting
    learning_rate=0.05,     # step size shrinkage — lower = more careful
    subsample=0.80,         # fraction of rows sampled per tree
    colsample_bytree=0.80,  # fraction of features sampled per tree
    eval_metric="auc",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=0,
    # scale_pos_weight is computed from the actual class distribution in y_train
    # — NOT hardcoded here — so it stays correct if the threshold changes.
)

# ── Colour palette for plots (one colour per model) ───────────────────────────
MODEL_COLORS = {
    "logistic_regression": "#2980b9",   # blue
    "random_forest":       "#27ae60",   # green
    "xgboost":             "#e67e22",   # orange
}

# ── Output paths ──────────────────────────────────────────────────────────────
OUTPUT_DIR      = Path("reports/outputs")
MODELS_DIR      = Path("models/saved")
PREDICTIONS_CSV = OUTPUT_DIR / "churn_predictions.csv"
COMPARISON_CSV  = OUTPUT_DIR / "model_comparison.csv"
DASHBOARD_PNG   = Path("reports/figures/churn_3model_comparison.png")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("✅  Settings loaded.")
print(f"    Churn threshold : {CHURN_THRESHOLD_DAYS} days")
print(f"    Features        : {len(FEATURE_COLUMNS)}")
print(f"    Test size       : {TEST_SIZE:.0%}")
print(f"    CV folds        : {CV_FOLDS}")

---
## Cell 3 — Load & Validate Data

Reads the transaction CSV and immediately checks:
- All required columns are present
- The date column parsed correctly as datetime
- Basic data quality (nulls, negative quantities)

**The pipeline fails here with a clear message if the data is wrong.**
This is intentional — catching problems early is far better than a
confusing crash deep inside model training.

In [ ]:
# =============================================================================
# LOAD DATA
# One row per transaction line-item in the raw CSV.
# =============================================================================

def load_data(file_path):
    """
    Load and validate the transaction CSV.

    ♻️  Reusability: update the SETTINGS cell column names instead of
    editing this function. The function reads column names from the
    SETTINGS variables, not from hardcoded strings.

    Parameters
    ----------
    file_path : str or Path
        Path to the CSV file.

    Returns
    -------
    pd.DataFrame
        Validated transaction-level DataFrame.
    """
    path = Path(file_path)

    # ── Validation 1: does the file exist? ────────────────────────────────────
    if not path.exists():
        raise FileNotFoundError(
            f"\n❌  CSV not found: {path.resolve()}"
            f"\n    Create the folder and place your CSV there,"
            f"\n    or update DATA_FILE in the SETTINGS cell."
        )

    print(f"📂  Loading: {path}")
    df = pd.read_csv(path, parse_dates=[DATE_COLUMN], low_memory=False)

    # ── Validation 2: required columns ────────────────────────────────────────
    required = {CUSTOMER_ID_COL, INVOICE_COL, DATE_COLUMN,
                PRODUCT_COL, QUANTITY_COL, REVENUE_COL, COUNTRY_COL}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(
            f"\n❌  CSV is missing these columns: {sorted(missing)}"
            f"\n    Columns found: {sorted(df.columns.tolist())}"
            f"\n    Update the column name variables in the SETTINGS cell."
        )

    # ── Validation 3: date column parsed correctly ────────────────────────────
    if not pd.api.types.is_datetime64_any_dtype(df[DATE_COLUMN]):
        raise ValueError(
            f"\n❌  '{DATE_COLUMN}' did not parse as dates."
            f"\n    Dates should be in YYYY-MM-DD or YYYY-MM-DD HH:MM:SS format."
        )

    # ── Data quality report ───────────────────────────────────────────────────
    print(f"\n{'='*55}")
    print("  DATA QUALITY REPORT")
    print(f"{'='*55}")
    print(f"  Total rows       : {len(df):>10,}")
    print(f"  Unique customers : {df[CUSTOMER_ID_COL].nunique():>10,}")
    print(f"  Unique invoices  : {df[INVOICE_COL].nunique():>10,}")
    print(f"  Date range       : {df[DATE_COLUMN].min().date()} → {df[DATE_COLUMN].max().date()}")

    # Warn on nulls
    null_pct = (df.isnull().mean() * 100).round(2)
    nulls = null_pct[null_pct > 0]
    if not nulls.empty:
        print(f"\n  ⚠️  Columns with missing values:")
        for col, pct in nulls.items():
            print(f"      {col}: {pct}%")
    else:
        print(f"  Null values      : none ✅")

    # Warn on negative quantities (returns/cancellations)
    neg_qty = (df[QUANTITY_COL] < 0).sum()
    if neg_qty > 0:
        print(f"  ⚠️  Negative quantities: {neg_qty:,} rows (returns/cancellations)")

    print(f"{'='*55}")
    return df


# ── Run it ────────────────────────────────────────────────────────────────────
transactions = load_data(DATA_FILE)
transactions.head()

---
## Cell 4 — Feature Engineering

Transforms **one row per transaction** → **one row per customer**.

### The 10 features and why they matter
| Feature | Business meaning |
|---------|-----------------|
| `TotalRevenue` | How much has this customer spent in total? |
| `TotalOrders` | How many times have they shopped? |
| `TotalItems` | How much do they buy per visit? |
| `AvgLineItemRevenue` | What price points do they buy at? |
| `BasketValue` | Average spend per trip |
| `RevenuePerItem` | Premium vs budget buyer signal |
| `UniqueProducts` | How broadly do they engage with the catalogue? |
| `UniqueCountries` | Are they a multi-country (business?) buyer? |
| `Tenure` | How long have they been a customer? |
| `AvgDaysBetween` | How frequently do they return? |

> **Why two DataFrames?** Returning `features` and `labels` separately
> makes it physically impossible to accidentally include `Recency` in the
> model — the most critical data leakage prevention step in the pipeline.

In [ ]:
# =============================================================================
# FEATURE ENGINEERING
# Aggregates transaction rows into one behavioural summary row per customer.
# Also computes the churn label — kept SEPARATE from features.
# =============================================================================

def build_features(df):
    """
    Build customer-level features and churn labels from transaction data.

    ♻️  Reusability: this function reads column names from the SETTINGS
    variables (CUSTOMER_ID_COL, INVOICE_COL, etc.) so it works on any
    dataset where you set those variables correctly.

    Parameters
    ----------
    df : pd.DataFrame
        Transaction-level DataFrame from load_data().

    Returns
    -------
    features : pd.DataFrame
        One row per customer. Columns: CustomerID + FEATURE_COLUMNS.
    labels : pd.DataFrame
        One row per customer. Columns: CustomerID, LastPurchaseDate,
        Recency, Churned.
        ⚠️  Do NOT merge Recency into the feature matrix.
    """
    # The "as-of" date — everything is measured relative to this date
    snapshot_date = df[DATE_COLUMN].max()

    print(f"\n{'='*55}")
    print("  FEATURE ENGINEERING")
    print(f"{'='*55}")
    print(f"  Snapshot date    : {snapshot_date.date()}")
    print(f"  Churn threshold  : {CHURN_THRESHOLD_DAYS} days")

    # ── Step 1: Aggregate — one row per customer ──────────────────────────────
    agg = df.groupby(CUSTOMER_ID_COL, sort=False).agg(
        TotalRevenue       = (REVENUE_COL,  "sum"),
        TotalOrders        = (INVOICE_COL,  "nunique"),  # distinct trips
        TotalItems         = (QUANTITY_COL, "sum"),
        AvgLineItemRevenue = (REVENUE_COL,  "mean"),     # per line, not per order
        UniqueProducts     = (PRODUCT_COL,  "nunique"),
        UniqueCountries    = (COUNTRY_COL,  "nunique"),
        FirstPurchaseDate  = (DATE_COLUMN,  "min"),
        LastPurchaseDate   = (DATE_COLUMN,  "max"),
    ).reset_index()

    # ── Step 2: Derived behavioural features ──────────────────────────────────

    # How long has this customer been with us? (first → last purchase)
    agg["Tenure"] = (agg["LastPurchaseDate"] - agg["FirstPurchaseDate"]).dt.days

    # Average spend per shopping trip
    agg["BasketValue"] = agg["TotalRevenue"] / agg["TotalOrders"].clip(lower=1)

    # Average price paid per unit
    agg["RevenuePerItem"] = agg["TotalRevenue"] / agg["TotalItems"].clip(lower=1)

    # Typical gap between purchases.
    # Problem: customers with only ONE order have Tenure=0, giving a
    # misleading 0-day interval (implies they shop every day).
    # Solution: impute single-order customers with the population median.
    multi_order_mask = agg["TotalOrders"] > 1
    agg["AvgDaysBetween"] = np.where(
        multi_order_mask,
        agg["Tenure"] / (agg["TotalOrders"] - 1).clip(lower=1),
        np.nan,   # placeholder for single-order customers
    )
    median_gap = agg["AvgDaysBetween"].median()
    n_imputed  = (~multi_order_mask).sum()
    agg["AvgDaysBetween"] = agg["AvgDaysBetween"].fillna(median_gap)

    # ── Step 3: Churn label ───────────────────────────────────────────────────
    # Days since last purchase — this IS the leakage risk.
    agg["Recency"] = (snapshot_date - agg["LastPurchaseDate"]).dt.days
    # Churned = 1 if inactive longer than the threshold
    agg["Churned"] = (agg["Recency"] > CHURN_THRESHOLD_DAYS).astype(int)

    # ── Step 4: Validate — no nulls allowed in features ───────────────────────
    # A null feature would either crash training or silently produce NaN predictions.
    null_counts = agg[FEATURE_COLUMNS].isnull().sum()
    if null_counts.sum() > 0:
        raise ValueError(
            f"\n❌  Feature matrix contains unexpected nulls:\n{null_counts[null_counts > 0]}"
        )

    # ── Step 5: Split into features and labels (the leakage barrier) ──────────
    features = agg[[CUSTOMER_ID_COL] + FEATURE_COLUMNS].copy()
    labels   = agg[[CUSTOMER_ID_COL, "LastPurchaseDate", "Recency", "Churned"]].copy()

    # ── Summary ───────────────────────────────────────────────────────────────
    churn_rate = labels["Churned"].mean()
    print(f"  Customers        : {len(features):,}")
    print(f"  Churn rate       : {churn_rate:.1%}  "
          f"({labels['Churned'].sum():,} churned / {(labels['Churned']==0).sum():,} active)")
    print(f"  Single-order     : {n_imputed:,} customers (AvgDaysBetween imputed "
          f"with median: {median_gap:.0f} days)")
    print(f"{'='*55}")

    return features, labels


# ── Run it ────────────────────────────────────────────────────────────────────
customer_features, customer_labels = build_features(transactions)

print("\nFeature matrix (first 5 rows):")
customer_features.head()

---
## Cell 5 — Train / Test Split

Divides customers into:
- **Training set (80%)** — what the models learn from
- **Test set (20%)** — used once at the very end to evaluate real-world performance

> **Why `stratify=y`?** Without it, the 33% churn rate might not be preserved
> in both halves. Stratification guarantees the same class balance in both splits.

In [ ]:
# =============================================================================
# TRAIN / TEST SPLIT
# The test set is never touched again until the final evaluation step.
# =============================================================================

# Separate inputs (X) from the target label (y)
X = customer_features[FEATURE_COLUMNS]  # shape: (n_customers, 10)
y = customer_labels["Churned"]           # shape: (n_customers,)  — 0 or 1

# Split: 80% train, 20% test
# stratify=y ensures the churn rate is the same in both splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"\n{'='*55}")
print("  TRAIN / TEST SPLIT")
print(f"{'='*55}")
print(f"  Training set  : {len(X_train):,} customers  (churn rate: {y_train.mean():.1%})")
print(f"  Test set      : {len(X_test):,} customers  (churn rate: {y_test.mean():.1%})")
print(f"  Features      : {X_train.shape[1]}")
print(f"\n  ✅  Churn rate preserved in both splits (stratify=y)")
print(f"{'='*55}")

---
## Cell 6 — Train Three Models

Three different model types are trained and compared:

| Model | Type | Why |
|-------|------|-----|
| Logistic Regression | Linear baseline | Fast, interpretable, needs scaling |
| Random Forest | Tree ensemble | Robust to outliers, no scaling needed |
| XGBoost | Gradient boosting | State-of-the-art on tabular data |

Each model is cross-validated on the **training set only** — the test set
is never seen during this step.

In [ ]:
# =============================================================================
# TRAIN THREE MODELS WITH 5-FOLD CROSS-VALIDATION
# =============================================================================

def train_models(X_train, y_train):
    """
    Build, cross-validate, and fit all three classification models.

    ♻️  Reusability: hyperparameters come from the SETTINGS cell.
    To add a fourth model: add its params to SETTINGS and add a new
    entry in the `model_definitions` dict below.

    Parameters
    ----------
    X_train : pd.DataFrame  — training feature matrix
    y_train : pd.Series     — training churn labels (0/1)

    Returns
    -------
    dict : { model_key: {"model": fitted, "cv_scores": array,
                         "train_time": float, "display_name": str} }
    """
    # ── XGBoost class imbalance correction ────────────────────────────────────
    # scale_pos_weight = (# negative samples) / (# positive samples)
    # This tells XGBoost to treat each churned customer as more important.
    # Computed from the actual training distribution — not hardcoded.
    n_neg = int((y_train == 0).sum())
    n_pos = int((y_train == 1).sum())
    spw   = n_neg / n_pos  # e.g. 2.0 if there are 2 active per 1 churned

    # ── Model definitions ─────────────────────────────────────────────────────
    # Logistic Regression is wrapped in a Pipeline with StandardScaler because
    # it is sensitive to feature magnitude. Trees don't need scaling.
    model_definitions = {
        "logistic_regression": (
            "Logistic Regression",
            Pipeline([
                ("scaler", StandardScaler()),
                ("logistic_regression", LogisticRegression(**LR_PARAMS)),
            ])
        ),
        "random_forest": (
            "Random Forest",
            RandomForestClassifier(**RF_PARAMS)
        ),
        "xgboost": (
            "XGBoost",
            XGBClassifier(**XGB_PARAMS, scale_pos_weight=spw)
        ),
    }

    cv      = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    results = {}

    print(f"\n{'='*55}")
    print("  MODEL TRAINING")
    print(f"{'='*55}")
    print(f"  Class distribution — active: {n_neg:,} | churned: {n_pos:,}")
    print(f"  XGBoost scale_pos_weight: {spw:.2f}")
    print()

    for key, (display_name, estimator) in model_definitions.items():

        # Step A: Cross-validate on training set
        # (5 folds — each fold validates on 20% of training data)
        print(f"  [{display_name}]  Cross-validating ({CV_FOLDS}-fold)...", end=" ")
        cv_scores = cross_val_score(
            estimator, X_train, y_train,
            cv=cv, scoring="roc_auc", n_jobs=-1,
        )
        print(f"CV AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

        # Step B: Fit on the full training set
        print(f"  [{display_name}]  Fitting on full training set...", end=" ")
        t0 = time.perf_counter()
        estimator.fit(X_train, y_train)
        elapsed = time.perf_counter() - t0
        print(f"done in {elapsed:.2f}s")

        results[key] = {
            "model":        estimator,
            "cv_scores":    cv_scores,
            "train_time":   elapsed,
            "display_name": display_name,
        }

    print(f"{'='*55}")
    print("  ✅  All models trained.")
    return results


# ── Run it ────────────────────────────────────────────────────────────────────
trained = train_models(X_train, y_train)

---
## Cell 7 — Evaluate on the Test Set

Now we look at the test set **for the first (and only) time**.

### Metrics explained
| Metric | What it answers |
|--------|----------------|
| **ROC-AUC** | How well does the model rank churners above non-churners? (1.0 = perfect) |
| **Avg Precision** | Area under Precision-Recall curve — better for imbalanced classes |
| **Precision** | Of customers we flagged as churned, what % actually churned? |
| **Recall** | Of customers who actually churned, what % did we catch? |
| **F1** | Harmonic mean of Precision and Recall |

> **Business note:** Recall usually matters more than Precision in retention.
> Missing a real churner (False Negative) costs the full customer lifetime value.
> A false alarm (False Positive) costs only one wasted retention offer.

In [ ]:
# =============================================================================
# EVALUATE ALL MODELS ON THE HOLD-OUT TEST SET
# =============================================================================

def evaluate_models(trained, X_test, y_test):
    """
    Compute all evaluation metrics for every trained model on the test set.

    Parameters
    ----------
    trained  : dict  — output of train_models()
    X_test   : pd.DataFrame
    y_test   : pd.Series

    Returns
    -------
    dict : { model_key: {"proba": array, "preds": array, "metrics": dict} }
    """
    y_test = y_test.reset_index(drop=True)
    evaluations = {}

    for key, info in trained.items():
        model = info["model"]
        # predict_proba returns [[p_active, p_churn], ...] — we want column 1
        proba = model.predict_proba(X_test)[:, 1]
        # predict returns hard 0/1 at the default 0.5 probability threshold
        preds = model.predict(X_test)

        evaluations[key] = {
            "proba": proba,
            "preds": preds,
            "metrics": {
                "test_auc":        roc_auc_score(y_test, proba),
                "avg_precision":   average_precision_score(y_test, proba),
                "precision_churn": precision_score(y_test, preds, zero_division=0),
                "recall_churn":    recall_score(y_test, preds, zero_division=0),
                "f1_churn":        f1_score(y_test, preds, zero_division=0),
            },
        }
    return evaluations


def build_comparison_table(trained, evaluations):
    """Build a tidy DataFrame with one row per model, all metrics as columns."""
    rows = []
    for key, info in trained.items():
        m = evaluations[key]["metrics"]
        rows.append({
            "Model":             info["display_name"],
            "CV AUC":            f"{info['cv_scores'].mean():.4f} ± {info['cv_scores'].std():.4f}",
            "Test AUC":          round(m["test_auc"], 4),
            "Avg Precision":     round(m["avg_precision"], 4),
            "Precision (Churn)": round(m["precision_churn"], 4),
            "Recall (Churn)":    round(m["recall_churn"], 4),
            "F1 (Churn)":        round(m["f1_churn"], 4),
            "Train Time (s)":    round(info["train_time"], 3),
        })
    return pd.DataFrame(rows).set_index("Model")


def select_best_model(trained, evaluations):
    """Return the (key, info) of the model with the highest test AUC."""
    best_key = max(evaluations, key=lambda k: evaluations[k]["metrics"]["test_auc"])
    return best_key, trained[best_key]


# ── Run it ────────────────────────────────────────────────────────────────────
evaluations   = evaluate_models(trained, X_test.reset_index(drop=True), y_test)
comparison_df = build_comparison_table(trained, evaluations)
best_key, best_info = select_best_model(trained, evaluations)

print(f"\n{'='*55}")
print("  MODEL EVALUATION — TEST SET RESULTS")
print(f"{'='*55}")
print(comparison_df.to_string())
print(f"\n  🏆  Best model : {best_info['display_name']}")
print(f"{'='*55}")

---
## Cell 8 — Score All Customers

Apply every trained model to **all** customers (not just the test set)
so the business receives a complete scored list.

Each customer gets:
- A churn probability from each individual model (0.0 → 1.0)
- An **Ensemble score** — the average across all three models
- A **Risk Tier** — Low / Medium / High based on the best model's probability

In [ ]:
# =============================================================================
# SCORE ALL CUSTOMERS AND ASSIGN RISK TIERS
# =============================================================================

def score_all_customers(customer_features, trained, best_key):
    """
    Score every customer with all trained models and assign a risk tier.

    Uses the FULL customer dataset (train + test) so the business
    receives a complete list, not just the held-out subset.

    Parameters
    ----------
    customer_features : pd.DataFrame — full customer feature matrix
    trained           : dict         — output of train_models()
    best_key          : str          — model key of the best model

    Returns
    -------
    pd.DataFrame : CustomerID, per-model probabilities, Ensemble score, RiskTier
                   Sorted descending by the best model's churn probability.
    """
    X_all  = customer_features[FEATURE_COLUMNS]
    scores = customer_features[[CUSTOMER_ID_COL]].copy()

    # Score with each model
    proba_cols = {}
    for key, info in trained.items():
        col = f"{info['display_name'].replace(' ', '_')}_ChurnProba"
        scores[col] = info["model"].predict_proba(X_all)[:, 1]
        proba_cols[key] = col

    # Ensemble score = simple average across all three models
    # Averaging reduces variance — when one model is overconfident, others compensate.
    scores["Ensemble_ChurnProba"] = scores[list(proba_cols.values())].mean(axis=1)

    # Risk tier from the best model's probability
    best_col = proba_cols[best_key]
    scores["RiskTier"] = pd.cut(
        scores[best_col],
        bins=RISK_BINS,
        labels=RISK_LABELS,
    )

    scores = scores.sort_values(best_col, ascending=False).reset_index(drop=True)

    print(f"\n{'='*55}")
    print("  RISK TIER DISTRIBUTION")
    print(f"{'='*55}")
    tier_counts = scores["RiskTier"].value_counts()
    for tier in ["High Risk", "Medium Risk", "Low Risk"]:
        if tier in tier_counts:
            count = tier_counts[tier]
            pct   = count / len(scores)
            bar   = "█" * int(pct * 30)
            print(f"  {tier:<14} : {count:>5,} ({pct:.1%})  {bar}")
    print(f"{'='*55}")

    return scores


# ── Run it ────────────────────────────────────────────────────────────────────
scored = score_all_customers(customer_features, trained, best_key)

# Attach true labels + business columns for the final report
scored = scored.merge(
    customer_labels[[CUSTOMER_ID_COL, "Churned", "Recency", "LastPurchaseDate"]],
    on=CUSTOMER_ID_COL, how="left",
)
scored = scored.merge(
    customer_features[[CUSTOMER_ID_COL, "TotalRevenue", "TotalOrders", "Tenure"]],
    on=CUSTOMER_ID_COL, how="left",
)

print("\nTop 10 highest-risk customers:")
best_col = f"{best_info['display_name'].replace(' ', '_')}_ChurnProba"
top_cols = [c for c in [CUSTOMER_ID_COL, "TotalRevenue", "Tenure",
                        best_col, "Ensemble_ChurnProba", "RiskTier"]
            if c in scored.columns]
scored[top_cols].head(10)

---
## Cell 9 — Save Models & Predictions

Three outputs are written to disk:
- **`.joblib` files** — the fitted models, loadable in milliseconds for future inference
- **`_metadata.json` files** — human-readable record of what each model was trained on
- **`churn_predictions.csv`** — all customers with scores and risk tiers
- **`model_comparison.csv`** — the metrics table

In [ ]:
# =============================================================================
# SAVE MODELS AND OUTPUTS TO DISK
# =============================================================================

def save_models(trained, evaluations):
    """
    Save every fitted model as .joblib + companion _metadata.json.

    The metadata JSON records feature columns, CV AUC, test AUC, and
    train time — making every saved model fully self-describing.
    """
    for key, info in trained.items():
        # Save the fitted model binary
        model_path = MODELS_DIR / f"{key}.joblib"
        joblib.dump(info["model"], model_path, compress=3)

        # Save human-readable metadata alongside it
        meta = {
            "model_key":          key,
            "display_name":       info["display_name"],
            "feature_columns":    FEATURE_COLUMNS,
            "churn_threshold_days": CHURN_THRESHOLD_DAYS,
            "cv_auc_mean":        round(float(info["cv_scores"].mean()), 6),
            "cv_auc_std":         round(float(info["cv_scores"].std()), 6),
            "train_time_seconds": round(info["train_time"], 4),
            "test_auc":           round(evaluations[key]["metrics"]["test_auc"], 6),
            "recall_churn":       round(evaluations[key]["metrics"]["recall_churn"], 6),
            "precision_churn":    round(evaluations[key]["metrics"]["precision_churn"], 6),
        }
        meta_path = MODELS_DIR / f"{key}_metadata.json"
        with meta_path.open("w") as f:
            json.dump(meta, f, indent=2)

        print(f"  ✅  Saved: {model_path.name}  +  {meta_path.name}")


# ── Run it ────────────────────────────────────────────────────────────────────
print("Saving models...")
save_models(trained, evaluations)

print("\nSaving prediction outputs...")
scored.to_csv(PREDICTIONS_CSV, index=False)
print(f"  ✅  {PREDICTIONS_CSV}")

comparison_df.to_csv(COMPARISON_CSV)
print(f"  ✅  {COMPARISON_CSV}")

---
## Cell 10 — 9-Panel Visualisation Dashboard

Generates a publication-ready comparison dashboard with:

| Row | Panels |
|-----|--------|
| 1 | ROC curves for all models  ·  Metric bar chart  ·  Training times |
| 2 | Feature importance: LR  ·  RF  ·  XGBoost |
| 3 | Confusion matrix: LR  ·  RF  ·  XGBoost |

In [ ]:
# =============================================================================
# 9-PANEL VISUAL COMPARISON DASHBOARD
# =============================================================================

def plot_dashboard(trained, evaluations, comparison_df, y_test, save_path=None):
    """
    Generate the full 9-panel model comparison dashboard.

    ♻️  Reusability: MODEL_COLORS in the SETTINGS cell controls panel colours.
    The function reads display names from the trained dict — no hardcoding.

    Parameters
    ----------
    trained       : dict         — output of train_models()
    evaluations   : dict         — output of evaluate_models()
    comparison_df : pd.DataFrame — output of build_comparison_table()
    y_test        : pd.Series    — true test labels
    save_path     : Path or None — if set, saves PNG to this path
    """
    y_test     = y_test.reset_index(drop=True)
    model_keys = list(trained.keys())

    sns.set_theme(style="whitegrid", font_scale=1.0)
    plt.rcParams.update({"figure.dpi": 130, "savefig.dpi": 150})

    fig = plt.figure(figsize=(22, 17))
    fig.suptitle(
        "Churn Prediction — 3-Model Comparison Dashboard",
        fontsize=15, fontweight="bold", y=0.99,
    )

    ax_roc    = fig.add_subplot(3, 3, 1)
    ax_metric = fig.add_subplot(3, 3, 2)
    ax_time   = fig.add_subplot(3, 3, 3)
    axes_feat = [fig.add_subplot(3, 3, i) for i in [4, 5, 6]]
    axes_cm   = [fig.add_subplot(3, 3, i) for i in [7, 8, 9]]

    # ── Panel 1: ROC curves ───────────────────────────────────────────────────
    for key in model_keys:
        fpr, tpr, _ = roc_curve(y_test, evaluations[key]["proba"])
        auc = evaluations[key]["metrics"]["test_auc"]
        ax_roc.plot(fpr, tpr, lw=2.5, color=MODEL_COLORS[key],
                    label=f"{trained[key]['display_name']} (AUC={auc:.4f})")
    ax_roc.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.35, label="Random baseline")
    ax_roc.set_title("ROC Curves — All Models", fontweight="bold", fontsize=9)
    ax_roc.set_xlabel("False Positive Rate", fontsize=8)
    ax_roc.set_ylabel("True Positive Rate", fontsize=8)
    ax_roc.legend(fontsize=7, loc="lower right")
    ax_roc.spines[["top", "right"]].set_visible(False)

    # ── Panel 2: Metric bar chart ─────────────────────────────────────────────
    metric_cols  = ["Test AUC", "Avg Precision", "Precision (Churn)",
                    "Recall (Churn)", "F1 (Churn)"]
    short_labels = ["AUC", "Avg Prec", "Precision", "Recall", "F1"]
    x = np.arange(len(metric_cols))
    bar_w = 0.25
    for i, key in enumerate(model_keys):
        vals = [comparison_df.loc[trained[key]["display_name"], col]
                for col in metric_cols]
        ax_metric.bar(x + i * bar_w, vals, bar_w,
                      label=trained[key]["display_name"],
                      color=MODEL_COLORS[key], alpha=0.85, edgecolor="white")
    all_vals = comparison_df[metric_cols].values.flatten()
    ax_metric.set_ylim(max(0, all_vals.min() - 0.05), min(1.0, all_vals.max() + 0.05))
    ax_metric.set_title("Performance Metrics Comparison", fontweight="bold", fontsize=9)
    ax_metric.set_xticks(x + bar_w)
    ax_metric.set_xticklabels(short_labels, fontsize=8)
    ax_metric.set_ylabel("Score", fontsize=8)
    ax_metric.legend(fontsize=7)
    ax_metric.spines[["top", "right"]].set_visible(False)

    # ── Panel 3: Training times ───────────────────────────────────────────────
    names  = [trained[k]["display_name"] for k in model_keys]
    times  = [trained[k]["train_time"] for k in model_keys]
    colors = [MODEL_COLORS[k] for k in model_keys]
    bars = ax_time.barh(names, times, color=colors, alpha=0.85, edgecolor="white")
    for bar, t in zip(bars, times):
        ax_time.text(bar.get_width() + max(times) * 0.02,
                     bar.get_y() + bar.get_height() / 2,
                     f"{t:.3f}s", va="center", fontsize=9)
    ax_time.set_title("Training Time (seconds)", fontweight="bold", fontsize=9)
    ax_time.set_xlabel("Seconds", fontsize=8)
    ax_time.spines[["top", "right"]].set_visible(False)

    # ── Panels 4-6: Feature importance ───────────────────────────────────────
    for ax, key in zip(axes_feat, model_keys):
        try:
            if key == "logistic_regression":
                raw = np.abs(trained[key]["model"].named_steps["logistic_regression"].coef_[0])
            else:
                raw = trained[key]["model"].feature_importances_

            normed = raw / raw.sum() if raw.sum() > 0 else raw
            imp_df = pd.DataFrame({"Feature": FEATURE_COLUMNS, "Importance": normed})
            imp_df = imp_df.sort_values("Importance", ascending=True)

            ax.barh(imp_df["Feature"], imp_df["Importance"],
                    color=MODEL_COLORS[key], alpha=0.8, edgecolor="white")
            ax.set_title(f"{trained[key]['display_name']} — Feature Importance",
                         fontweight="bold", fontsize=9)
            ax.set_xlabel("Normalised Importance", fontsize=8)
            ax.tick_params(axis="y", labelsize=8)
            ax.spines[["top", "right"]].set_visible(False)
        except Exception:
            ax.text(0.5, 0.5, "N/A", ha="center", va="center", transform=ax.transAxes)

    # ── Panels 7-9: Confusion matrices ───────────────────────────────────────
    for ax, key in zip(axes_cm, model_keys):
        cm   = confusion_matrix(y_test, evaluations[key]["preds"])
        disp = ConfusionMatrixDisplay(cm, display_labels=["Active", "Churned"])
        disp.plot(ax=ax, colorbar=False, cmap="Blues")
        ax.set_title(f"{trained[key]['display_name']} — Confusion Matrix",
                     fontweight="bold", fontsize=9)

    plt.tight_layout(rect=[0, 0, 1, 0.97])

    if save_path:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"  ✅  Dashboard saved: {save_path}")

    plt.show()


# ── Run it ────────────────────────────────────────────────────────────────────
plot_dashboard(trained, evaluations, comparison_df, y_test, save_path=DASHBOARD_PNG)

---
## Cell 11 — Churn Probability Distribution

A well-calibrated model produces **two clearly separated humps**:
- Active customers clustered near 0
- Churned customers clustered near 1

Overlap between the humps shows where the model is genuinely uncertain.

In [ ]:
# =============================================================================
# CHURN PROBABILITY DISTRIBUTION PLOT
# =============================================================================

def plot_probability_distribution(scored, best_key, trained):
    """
    Overlay histogram of churn probabilities split by true label.
    Shows how well the best model separates churned from active customers.
    """
    best_col = f"{trained[best_key]['display_name'].replace(' ', '_')}_ChurnProba"

    fig, ax = plt.subplots(figsize=(10, 5))

    for label, colour, name in [
        (0, "#27ae60", "Active  (label = 0)"),
        (1, "#e74c3c", "Churned (label = 1)"),
    ]:
        subset = scored.loc[scored["Churned"] == label, best_col]
        ax.hist(subset, bins=40, alpha=0.6, color=colour,
                label=f"{name}  —  n={len(subset):,}",
                edgecolor="white", linewidth=0.4)

    ax.set_title(
        f"Predicted Churn Probability Distribution — {trained[best_key]['display_name']}",
        fontweight="bold"
    )
    ax.set_xlabel("Predicted Churn Probability")
    ax.set_ylabel("Customer Count")
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.legend(fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.show()


# ── Run it ────────────────────────────────────────────────────────────────────
plot_probability_distribution(scored, best_key, trained)

---
## Cell 12 — Final Business Summary

The complete pipeline results in one clean printout —
the table a stakeholder would see in a Monday morning report.

In [ ]:
# =============================================================================
# FINAL BUSINESS SUMMARY REPORT
# =============================================================================

def print_summary(trained, evaluations, comparison_df, scored, best_key):
    """Print the complete pipeline results as a formatted terminal report."""
    n          = len(scored)
    churn_rate = scored["Churned"].mean()
    sep        = "=" * 65

    print(f"\n{sep}")
    print("  FINAL SUMMARY — CHURN PREDICTION PIPELINE")
    print(sep)
    print(f"  Churn definition : no purchase in > {CHURN_THRESHOLD_DAYS} days")
    print(f"  Overall churn rate : {churn_rate:.1%}")
    print(f"  Total customers    : {n:,}")
    print(f"  Features used      : {len(FEATURE_COLUMNS)}")

    print(f"\n  CROSS-VALIDATION AUC ({CV_FOLDS}-fold, training set only):")
    for key, info in trained.items():
        cv = info["cv_scores"]
        print(f"    {info['display_name']:<24} : {cv.mean():.4f} ± {cv.std():.4f}")

    print(f"\n  HOLD-OUT TEST SET METRICS:")
    cols = ["Test AUC", "Precision (Churn)", "Recall (Churn)", "F1 (Churn)", "Train Time (s)"]
    print(comparison_df[cols].to_string())

    print(f"\n  🏆  Best model : {trained[best_key]['display_name']}")

    print(f"\n  RISK TIER BREAKDOWN (based on {trained[best_key]['display_name']}):")
    tier_counts = scored["RiskTier"].value_counts()
    for tier in ["High Risk", "Medium Risk", "Low Risk"]:
        if tier in tier_counts:
            count = tier_counts[tier]
            pct   = count / n
            bar   = "█" * int(pct * 35)
            print(f"    {tier:<14} : {count:>5,}  ({pct:.1%})  {bar}")

    print(f"\n  TOP 10 HIGHEST-RISK CUSTOMERS:")
    best_col = f"{trained[best_key]['display_name'].replace(' ', '_')}_ChurnProba"
    top_cols = [c for c in [CUSTOMER_ID_COL, "TotalRevenue", "Tenure",
                             best_col, "Ensemble_ChurnProba", "RiskTier"]
                if c in scored.columns]
    print(scored[top_cols].head(10).to_string(index=False))

    print(f"\n  OUTPUT FILES:")
    print(f"    {PREDICTIONS_CSV}")
    print(f"    {COMPARISON_CSV}")
    print(f"    {DASHBOARD_PNG}")
    print(f"    {MODELS_DIR}/  (3 x .joblib + 3 x _metadata.json)")
    print(sep)


# ── Run it ────────────────────────────────────────────────────────────────────
print_summary(trained, evaluations, comparison_df, scored, best_key)

---
## Cell 13 — ♻️  Load Saved Model for Future Predictions

Once trained, you can reload any saved model and score new customers
**without retraining**. This cell shows exactly how.

In [ ]:
# =============================================================================
# LOAD A SAVED MODEL AND SCORE NEW CUSTOMERS (no retraining needed)
# =============================================================================

def load_saved_model(model_key):
    """
    Load a previously saved model and its metadata from disk.

    Parameters
    ----------
    model_key : str — e.g. 'random_forest', 'xgboost', 'logistic_regression'

    Returns
    -------
    model    : fitted sklearn estimator
    metadata : dict with feature columns, AUC scores, training info
    """
    model_path = MODELS_DIR / f"{model_key}.joblib"
    meta_path  = MODELS_DIR / f"{model_key}_metadata.json"

    if not model_path.exists():
        raise FileNotFoundError(
            f"No saved model found at {model_path}\n"
            "Run Cell 9 (save_models) first."
        )

    model = joblib.load(model_path)
    with meta_path.open() as f:
        metadata = json.load(f)

    print(f"✅  Loaded: {model_key}")
    print(f"    Features : {metadata['feature_columns']}")
    print(f"    Test AUC : {metadata['test_auc']}")
    print(f"    Trained  : {metadata.get('train_time_seconds', 'n/a')}s")
    return model, metadata


def predict_new_customers(new_transactions_df, model_key="random_forest"):
    """
    Score a new batch of customers using a saved model.

    Parameters
    ----------
    new_transactions_df : pd.DataFrame
        Raw transaction data in the same format as the original CSV.
        Must have the same columns as defined in SETTINGS.
    model_key : str
        Which saved model to use.

    Returns
    -------
    pd.DataFrame : CustomerID, ChurnProbability, RiskTier
    """
    # Build features from the new transactions (same function as before)
    new_features, _ = build_features(new_transactions_df)

    # Load the saved model
    model, metadata = load_saved_model(model_key)

    # Verify feature columns match what the model was trained on
    trained_features = metadata["feature_columns"]
    if trained_features != FEATURE_COLUMNS:
        print(f"⚠️  Feature mismatch! Model was trained on: {trained_features}")

    # Score
    X_new = new_features[trained_features]
    proba = model.predict_proba(X_new)[:, 1]

    results = new_features[[CUSTOMER_ID_COL]].copy()
    results["ChurnProbability"] = proba.round(4)
    results["RiskTier"] = pd.cut(proba, bins=RISK_BINS, labels=RISK_LABELS)
    results = results.sort_values("ChurnProbability", ascending=False).reset_index(drop=True)

    print(f"\n✅  Scored {len(results):,} new customers.")
    return results


# ── Demo: reload the best model and verify it works ───────────────────────────
print("Reloading best model from disk to verify save/load works...")
reloaded_model, reloaded_meta = load_saved_model(best_key)

# Quick sanity check — predictions should match what we computed above
X_check   = X_test.reset_index(drop=True)
reloaded_proba = reloaded_model.predict_proba(X_check)[:, 1]
original_proba = evaluations[best_key]["proba"]
match = np.allclose(reloaded_proba, original_proba, atol=1e-6)
print(f"\n  Predictions match original: {'✅  Yes' if match else '❌  No'}")
print("\n  To score new data in future:")
print("  >>> new_scores = predict_new_customers(new_transactions_df, model_key='random_forest')")

---
## ✅  Pipeline Complete

| Output | Location |
|--------|----------|
| All customer churn scores | `reports/outputs/churn_predictions.csv` |
| Model comparison table | `reports/outputs/model_comparison.csv` |
| 9-panel visual dashboard | `reports/figures/churn_3model_comparison.png` |
| Saved models (3×) | `models/saved/*.joblib` |
| Model metadata (3×) | `models/saved/*_metadata.json` |

---

### ♻️  Quick Adaptation Checklist

To run this on a **different dataset**:

- [ ] Update `DATA_FILE`, `DATE_COLUMN`, `CUSTOMER_ID_COL`, `INVOICE_COL`,
      `REVENUE_COL`, `QUANTITY_COL`, `PRODUCT_COL`, `COUNTRY_COL` in Cell 2
- [ ] Adjust `CHURN_THRESHOLD_DAYS` to match your business definition
- [ ] Review `FEATURE_COLUMNS` — remove any that don't exist in your data
- [ ] Run all cells top to bottom (`Kernel → Restart & Run All`)

> The ML logic, visualisations, and reporting work unchanged on any
> transaction dataset that has customers, orders, dates, and revenue.